In [6]:
from pathlib import Path

def find_base(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "raw").exists():
            return p
    raise FileNotFoundError("No encuentro la carpeta 'raw' en ningún parent de la ruta actual")

BASE = find_base(Path.cwd())
RAW  = BASE / "raw"
OUT  = BASE / "processed" / "chunks_legal"
OUT.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("RAW exists:", RAW.exists(), RAW)
print("OUT exists:", OUT.exists(), OUT)

print("BOE files:", len(list((RAW / "boe").glob("*"))))
print("EU files:", len(list((RAW).glob("EU AI Act completo*/*"))))
print("AESIA PDF files:", len(list((RAW / "Guías AESIA + sandbox regulatorio").glob("*.pdf"))))

BASE: /Users/danielocando/IA/KeepCoding/BOOTCAMP IA/NormaBot TFM BOOTCAMP IA IV/NormaBoot_Data
RAW exists: True /Users/danielocando/IA/KeepCoding/BOOTCAMP IA/NormaBot TFM BOOTCAMP IA IV/NormaBoot_Data/raw
OUT exists: True /Users/danielocando/IA/KeepCoding/BOOTCAMP IA/NormaBot TFM BOOTCAMP IA IV/NormaBoot_Data/processed/chunks_legal
BOE files: 3
EU files: 1
AESIA PDF files: 16


In [ ]:
# 1) Leer BOE (HTML) -> texto plano
from bs4 import BeautifulSoup

boe_dir = RAW / "boe"
boe_files = sorted(boe_dir.glob("*.html"))

boe_texts = []
for fp in boe_files:
    html = fp.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text("\n")
    text = "\n".join(line.strip() for line in text.splitlines() if line.strip())
    boe_texts.append({"source": "boe", "file": fp.name, "text": text})

len(boe_texts), boe_texts[0]["file"] if boe_texts else None

(3, 'BOE-A-2012-1674.html')

In [8]:
import json

out_file = OUT / "boe_texts.jsonl"

with out_file.open("w", encoding="utf-8") as f:
    for row in boe_texts:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

out_file, out_file.exists(), sum(1 for _ in out_file.open("r", encoding="utf-8"))

(PosixPath('/Users/danielocando/IA/KeepCoding/BOOTCAMP IA/NormaBot TFM BOOTCAMP IA IV/NormaBoot_Data/processed/chunks_legal/boe_texts.jsonl'),
 True,
 3)

In [12]:
# 2) Leer EU AI Act (HTML) -> texto plano + guardar JSONL
import json
from bs4 import BeautifulSoup

eu_dir = RAW / "EU AI Act completo (Reglamento UE 2024-1689)"
eu_files = sorted(eu_dir.glob("*.html"))

eu_texts = []
for fp in eu_files:
    html = fp.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text("\n")
    text = "\n".join(line.strip() for line in text.splitlines() if line.strip())
    eu_texts.append({"source": "eu_ai_act", "file": fp.name, "text": text})

out_file = OUT / "eu_ai_act_texts.jsonl"
with out_file.open("w", encoding="utf-8") as f:
    for row in eu_texts:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

len(eu_texts), eu_texts[0]["file"] if eu_texts else None, out_file.name

(1, 'EU_AI_Act_2024_1689_ES.html', 'eu_ai_act_texts.jsonl')

In [13]:
# 3) Leer AESIA (PDF) -> texto plano + guardar JSONL
import json
from pathlib import Path
from pypdf import PdfReader

aes_dir = RAW / "Guías AESIA + sandbox regulatorio"
pdf_files = sorted(aes_dir.glob("*.pdf"))

aes_texts = []
for fp in pdf_files:
    reader = PdfReader(str(fp))
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        t = page.extract_text() or ""
        t = "\n".join(line.strip() for line in t.splitlines() if line.strip())
        if t:
            pages.append({"page": i, "text": t})
    full_text = "\n\n".join(p["text"] for p in pages)
    aes_texts.append({
        "source": "aesia",
        "file": fp.name,
        "n_pages": len(reader.pages),
        "pages_extracted": len(pages),
        "text": full_text
    })

out_file = OUT / "aesia_texts.jsonl"
with out_file.open("w", encoding="utf-8") as f:
    for row in aes_texts:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

len(aes_texts), aes_texts[0]["file"] if aes_texts else None, out_file.name

(16,
 '02-guia-practica-y-ejemplos-para-entender-el-reglamento-de-ia.pdf',
 'aesia_texts.jsonl')

In [14]:
# 4) Chunking: leer boe/eu/aesia JSONL -> trocear -> guardar chunks.jsonl
import json, re, hashlib
from pathlib import Path

# uso variables BASE/OUT/RAW 
in_files = [
    OUT / "boe_texts.jsonl",
    OUT / "eu_ai_act_texts.jsonl",
    OUT / "aesia_texts.jsonl",   
]


if not (OUT / "aesia_texts.jsonl").exists() and (OUT / "aesia_texts.jsonl").exists():
    in_files[2] = OUT / "aesia_texts.jsonl"

def normalize_ws(s: str) -> str:
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def split_paragraphs(text: str):
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

def chunk_text(text: str, chunk_chars=2500, overlap_chars=300):
    text = normalize_ws(text)
    paras = split_paragraphs(text)
    chunks = []
    buf = ""
    for p in paras:
        if not buf:
            buf = p
        elif len(buf) + 2 + len(p) <= chunk_chars:
            buf = buf + "\n\n" + p
        else:
            chunks.append(buf)
            # overlap: conserva cola del chunk anterior
            tail = buf[-overlap_chars:] if overlap_chars and len(buf) > overlap_chars else ""
            buf = (tail + "\n\n" + p).strip()
    if buf:
        chunks.append(buf)
    return chunks

out_chunks = OUT / "chunks_all.jsonl"

total_docs = 0
total_chunks = 0

with out_chunks.open("w", encoding="utf-8") as fout:
    for fp in in_files:
        if not fp.exists():
            raise FileNotFoundError(f"No existe: {fp}")

        with fp.open("r", encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)
                source = row.get("source", "unknown")
                file_ = row.get("file", fp.name)
                text = row.get("text", "")

                total_docs += 1
                chunks = chunk_text(text, chunk_chars=2500, overlap_chars=300)

                for i, ch in enumerate(chunks):
                    chunk_id = hashlib.md5(f"{source}|{file_}|{i}|{ch[:200]}".encode("utf-8")).hexdigest()
                    out_row = {
                        "id": chunk_id,
                        "source": source,
                        "file": file_,
                        "chunk_index": i,
                        "text": ch,
                    }
                    fout.write(json.dumps(out_row, ensure_ascii=False) + "\n")
                    total_chunks += 1

total_docs, total_chunks, out_chunks.name

(20, 697, 'chunks_all.jsonl')

In [15]:
# Validar que chunks_all.jsonl tiene chunks “buenos” (texto + metadatos) antes de “chunking inteligente

import json
from pathlib import Path

fp = OUT / "chunks_all.jsonl"
print("EXISTS:", fp.exists(), fp)

# contar líneas (chunks)
n = sum(1 for _ in fp.open("r", encoding="utf-8"))
print("TOTAL_CHUNKS:", n)

# ver 1 chunk real (primero)
with fp.open("r", encoding="utf-8") as f:
    first = json.loads(f.readline())
print("KEYS:", list(first.keys()))
print("SOURCE:", first.get("source"), "| FILE:", first.get("file"), "| IDX:", first.get("chunk_index"))
print("TEXT_PREVIEW:", (first.get("text","")[:300]).replace("\n"," "))

EXISTS: True /Users/danielocando/IA/KeepCoding/BOOTCAMP IA/NormaBot TFM BOOTCAMP IA IV/NormaBoot_Data/processed/chunks_legal/chunks_all.jsonl
TOTAL_CHUNKS: 697
KEYS: ['id', 'source', 'file', 'chunk_index', 'text']
SOURCE: boe | FILE: BOE-A-2012-1674.html | IDX: 0
TEXT_PREVIEW: BOE-A-2012-1674 Real Decreto-ley 2/2012, de 3 de febrero, de saneamiento del sector financiero. Agencia Estatal Boletín Oficial del Estado Ir a contenido Consultar el diario oficial BOE Idioma actual: Castellano / es Puede seleccionar otro idioma: es / Castellano ca / Català gl / Galego eu / Euskara


In [16]:
# 6) Chunking inteligente (por Artículo/Sección) + metadata rica -> chunks_structured.jsonl

import json, re, hashlib
from pathlib import Path

OUT.mkdir(parents=True, exist_ok=True)

boe_fp = OUT / "boe_texts.jsonl"
eu_fp  = OUT / "eu_ai_act_texts.jsonl"
aes_fp = OUT / "aesia_texts.jsonl"

out_fp = OUT / "chunks_structured.jsonl"

def md5(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def norm_spaces(s: str) -> str:
    s = s.replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def split_units(text: str, patterns):
    """
    patterns: lista de regex (con grupo de captura opcional) que detectan "cabeceras" de unidad.
    Devuelve lista de (header, body).
    """
    t = "\n" + text.strip() + "\n"
    hits = []
    for pat in patterns:
        for m in re.finditer(pat, t, flags=re.IGNORECASE | re.MULTILINE):
            hits.append((m.start(), m.group(0).strip()))
    hits = sorted(set(hits), key=lambda x: x[0])

    if not hits:
        return [("DOCUMENT", text.strip())]

    units = []
    for i, (pos, header) in enumerate(hits):
        start = pos
        end = hits[i+1][0] if i+1 < len(hits) else len(t)
        chunk = t[start:end].strip()
        # header = primera línea del chunk
        first_line = chunk.splitlines()[0].strip() if chunk else header
        body = "\n".join(chunk.splitlines()[1:]).strip()
        units.append((first_line, body if body else chunk))
    return units

def parse_doc_meta(source: str, file: str, text: str):
    title = None
    date = None

    # título: primera línea "fuerte"
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    if lines:
        title = lines[0][:200]

    # fechas típicas ES: "3 de febrero de 2012"
    m = re.search(r"\b(\d{1,2})\s+de\s+(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|setiembre|octubre|noviembre|diciembre)\s+de\s+(\d{4})\b", text, flags=re.IGNORECASE)
    if m:
        date = f"{m.group(1)} {m.group(2).lower()} {m.group(3)}"

    # BOE file: BOE-A-2012-1674.html -> year=2012, boe_id=1674
    boe_year = None
    boe_id = None
    if source == "boe":
        m2 = re.search(r"BOE-[A-Z]-([0-9]{4})-([0-9]+)", file)
        if m2:
            boe_year, boe_id = m2.group(1), m2.group(2)

    return {
        "doc_title": title,
        "doc_date": date,
        "boe_year": boe_year,
        "boe_id": boe_id,
    }

def unit_meta_from_header(source: str, header: str):
    h = header.strip()

    # defaults
    unit_type = "section"
    unit_id = None
    unit_title = h[:200]

    if re.search(r"\bart[íi]culo\b", h, flags=re.IGNORECASE):
        unit_type = "article"
        m = re.search(r"\bart[íi]culo\s+([0-9]+)\b", h, flags=re.IGNORECASE)
        if m:
            unit_id = m.group(1)

    elif re.search(r"\barticle\b", h, flags=re.IGNORECASE):
        unit_type = "article"
        m = re.search(r"\barticle\s+([0-9]+)\b", h, flags=re.IGNORECASE)
        if m:
            unit_id = m.group(1)

    elif re.search(r"\bcap[íi]tulo\b", h, flags=re.IGNORECASE):
        unit_type = "chapter"
        m = re.search(r"\bcap[íi]tulo\s+([ivx]+|[0-9]+)\b", h, flags=re.IGNORECASE)
        if m:
            unit_id = m.group(1)

    elif re.search(r"\bsecci[oó]n\b", h, flags=re.IGNORECASE):
        unit_type = "section"
        m = re.search(r"\bsecci[oó]n\s+([ivx]+|[0-9]+)\b", h, flags=re.IGNORECASE)
        if m:
            unit_id = m.group(1)

    return unit_type, unit_id, unit_title

def build_structured_chunks(source: str, texts_jsonl: Path, patterns):
    rows_out = []
    with texts_jsonl.open("r", encoding="utf-8") as f:
        for line in f:
            doc = json.loads(line)
            file = doc.get("file")
            text = norm_spaces(doc.get("text", ""))

            dmeta = parse_doc_meta(source, file, text)
            units = split_units(text, patterns)

            for u_idx, (header, body) in enumerate(units):
                body = norm_spaces(body)
                if len(body) < 80:
                    continue

                unit_type, unit_id, unit_title = unit_meta_from_header(source, header)

                payload = {
                    "source": source,
                    "file": file,
                    "doc_title": dmeta["doc_title"],
                    "doc_date": dmeta["doc_date"],
                    "boe_year": dmeta.get("boe_year"),
                    "boe_id": dmeta.get("boe_id"),
                    "unit_type": unit_type,
                    "unit_id": unit_id,
                    "unit_title": unit_title,
                    "unit_index": u_idx,
                    "text": body,
                }
                payload["id"] = md5(f'{source}|{file}|{unit_type}|{unit_id}|{u_idx}|{body[:200]}')
                rows_out.append(payload)
    return rows_out

# patrones por fuente
boe_patterns = [
    r"^\s*Art[íi]culo\s+\d+\.?\s*$",
    r"^\s*T[íi]TULO\s+[IVXLC0-9]+\b.*$",
    r"^\s*CAP[ÍI]TULO\s+[IVXLC0-9]+\b.*$",
    r"^\s*Secci[oó]n\s+[IVXLC0-9]+\b.*$",
]

eu_patterns = [
    r"^\s*Article\s+\d+\.?\s*$",
    r"^\s*Art[íi]culo\s+\d+\.?\s*$",
    r"^\s*CHAPTER\s+[IVXLC0-9]+\b.*$",
    r"^\s*CAP[ÍI]TULO\s+[IVXLC0-9]+\b.*$",
    r"^\s*SECTION\s+[IVXLC0-9]+\b.*$",
    r"^\s*Secci[oó]n\s+[IVXLC0-9]+\b.*$",
]

aes_patterns = [
    r"^\s*CAP[ÍI]TULO\s+[IVXLC0-9]+\b.*$",
    r"^\s*Secci[oó]n\s+[IVXLC0-9]+\b.*$",
    r"^\s*Art[íi]culo\s+\d+\.?\s*$",
]

rows = []
rows += build_structured_chunks("boe", boe_fp, boe_patterns)
rows += build_structured_chunks("eu_ai_act", eu_fp, eu_patterns)
rows += build_structured_chunks("aesia", aes_fp, aes_patterns)

with out_fp.open("w", encoding="utf-8") as fout:
    for r in rows:
        fout.write(json.dumps(r, ensure_ascii=False) + "\n")

len(rows), out_fp.name

(185, 'chunks_structured.jsonl')

In [19]:
# 6) Sanity check chunks_structured: mostrar 1 ejemplo por source
import json
from pathlib import Path

fp = OUT / "chunks_structured.jsonl"

def show_one(src: str):
    with fp.open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row.get("source") == src:
                print(f"\n--- {src} SAMPLE ---")
                print("file:", row.get("file"))
                print("doc_title:", row.get("doc_title"))
                print("doc_date:", row.get("doc_date"))
                txt = (row.get("text") or "").replace("\n", " ")
                print("text_len:", len(txt))
                print("preview:", txt[:250])
                return
    print(f"\n--- {src}: NO ENCONTRADO ---")

show_one("boe")
show_one("eu_ai_act")
show_one("aesia")


--- boe SAMPLE ---
file: BOE-A-2012-1674.html
doc_title: BOE-A-2012-1674 Real Decreto-ley 2/2012, de 3 de febrero, de saneamiento del sector financiero.
doc_date: 24 noviembre 2010
text_len: 17598
preview: Disposición adicional primera Disposición adicional segunda Disposición adicional tercera Disposición adicional cuarta Disposición adicional quinta Disposición derogatoria única Disposición final primera Disposición final segunda Disposición final te

--- eu_ai_act SAMPLE ---
file: EU_AI_Act_2024_1689_ES.html
doc_title: L_202401689ES.000101.fmx.xml
doc_date: 13 junio 2024
text_len: 1297
preview: Objeto 1.   El objetivo del presente Reglamento es mejorar el funcionamiento del mercado interior y promover la adopción de una inteligencia artificial (IA) centrada en el ser humano y fiable, garantizando al mismo tiempo un elevado nivel de protecci

--- aesia SAMPLE ---
file: 02-guia-practica-y-ejemplos-para-entender-el-reglamento-de-ia.pdf
doc_title: 0
doc_date: 10 diciembre 2025
text_len

In [20]:
# 7) Export final: chunks_structured.jsonl -> chunks_final.jsonl (con id + metadatos)

import json, hashlib
from pathlib import Path

inp = OUT / "chunks_structured.jsonl"
outp = OUT / "chunks_final.jsonl"

print("IN EXISTS:", inp.exists(), inp.name)

total = 0
with inp.open("r", encoding="utf-8") as fin, outp.open("w", encoding="utf-8") as fout:
    for line in fin:
        r = json.loads(line)
        txt = (r.get("text") or "").strip()
        if not txt:
            continue

        # id estable
        chunk_id = hashlib.md5(f"{r.get('source','')}|{r.get('file','')}|{r.get('chunk_index','')}|{txt}".encode("utf-8")).hexdigest()

        out_row = {
            "id": chunk_id,
            "source": r.get("source"),
            "file": r.get("file"),
            "doc_title": r.get("doc_title"),
            "doc_date": r.get("doc_date"),
            "boe_year": r.get("boe_year"),
            "boe_number": r.get("boe_number"),
            "boe_ref": r.get("boe_ref"),
            "eu_celex": r.get("eu_celex"),
            "section": r.get("section"),
            "article": r.get("article"),
            "chunk_index": r.get("chunk_index"),
            "text": txt,
        }
        fout.write(json.dumps(out_row, ensure_ascii=False) + "\n")
        total += 1

print("OUT WRITTEN:", total, outp.name)

IN EXISTS: True chunks_structured.jsonl
OUT WRITTEN: 185 chunks_final.jsonl


In [ ]:
# 8) Sanity check FINAL: chunks_final.jasonl ( conteo, schema, ejemplos por source )
import json
from collections import Counter

fp = OUT / "chunks_final.jsonl"
print("EXISTS:", fp.exists(), fp.name)

total = 0
by_source = Counter()
missing_text = 0
empty_text = 0

samples = {"boe": None, "eu_ai_act": None, "aesia": None}

with fp.open("r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        total += 1

        src = r.get("source")
        by_source[src] += 1

        txt = r.get("text")
        if txt is None:
            missing_text += 1
        elif not str(txt).strip():
            empty_text += 1

        if src in samples and samples[src] is None:
            samples[src] = r

print("TOTAL_ROWS:", total)
print("BY_SOURCE:", by_source)
print("MISSING_TEXT:", missing_text, "| EMPTY_TEXT:", empty_text)

def print_sample(src, r):
    print(f"\n--- SAMPLE: {src} ---")
    if not r:
        print("NO ENCONTRADO")
        return
    print("id:", r.get("id"))
    print("file:", r.get("file"))
    print("chunk_index:", r.get("chunk_index"))
    print("doc_title:", r.get("doc_title"))
    print("doc_date:", r.get("doc_date"))
    txt = (r.get("text") or "").replace("\n", " ")
    print("text_len:", len(r.get("text") or ""))
    print("preview:", txt[:500])

print_sample("boe", samples["boe"])
print_sample("eu_ai_act", samples["eu_ai_act"])
print_sample("aesia", samples["aesia"])

EXISTS: True chunks_final.jsonl
TOTAL_ROWS: 185
BY_SOURCE: Counter({'eu_ai_act': 121, 'boe': 38, 'aesia': 26})
MISSING_TEXT: 0 | EMPTY_TEXT: 0

--- SAMPLE: boe ---
id: f7c3e78d682b1ec25cce980817c981e6
file: BOE-A-2012-1674.html
chunk_index: None
doc_title: BOE-A-2012-1674 Real Decreto-ley 2/2012, de 3 de febrero, de saneamiento del sector financiero.
doc_date: 24 noviembre 2010
text_len: 17598
preview: Disposición adicional primera Disposición adicional segunda Disposición adicional tercera Disposición adicional cuarta Disposición adicional quinta Disposición derogatoria única Disposición final primera Disposición final segunda Disposición final tercera Disposición final cuarta Disposición final quinta Disposición final sexta Disposición final séptima [Firma] ANEXO I ANEXO II EXPOSICIÓN DE MOTIVOS I Cuatro años después del inicio de la crisis financiera internacional, los problemas de confianz

--- SAMPLE: eu_ai_act ---
id: b5238c58e74065ca5ac5b166b6a65546
file: EU_AI_Act_2024_1689_ES.

In [ ]:
# 9) Descargar LOPD-GDD (BOE) + RGPD (AEPD) a RAW/Normativa LOPD-GDD, RGPD
import urllib.request
from pathlib import Path

dst_dir = RAW / "Normativa LOPD-GDD, RGPD"
dst_dir.mkdir(parents=True, exist_ok=True)

files = [
    (
        "LOPDGDD_LO_3_2018_BOE.pdf",
        "https://www.boe.es/boe/dias/2018/12/06/pdfs/BOE-A-2018-16673.pdf",
    ),
    (
        "RGPD_AEPD_consolidado_ES.pdf",
        "https://www.aepd.es/documento/reglamento-ue-2016-679-consolidado.pdf",
    ),
]

for name, url in files:
    out = dst_dir / name
    print("DOWNLOADING:", name)
    urllib.request.urlretrieve(url, out)

print("DONE. Files:", sorted([p.name for p in dst_dir.iterdir()]))

DOWNLOADING: LOPDGDD_LO_3_2018_BOE.pdf
DOWNLOADING: RGPD_AEPD_consolidado_ES.pdf
DONE. Files: ['LOPDGDD_LO_3_2018_BOE.pdf', 'RGPD_AEPD_consolidado_ES.pdf']


In [25]:
# 10) LOPD/RGPD: leer PDFs -> texto plano + guardar JSONL

import json
from pathlib import Path

src_dir = RAW / "Normativa LOPD-GDD, RGPD"
out_fp  = OUT / "lopd_rgpd_texts.jsonl"

pdfs = sorted(src_dir.glob("*.pdf"))
print("PDFS:", [p.name for p in pdfs])

def extract_pdf_text(pdf_path: Path) -> str:
    try:
        import pdfplumber
        parts = []
        with pdfplumber.open(str(pdf_path)) as pdf:
            for page in pdf.pages:
                parts.append(page.extract_text() or "")
        return "\n".join(parts)
    except Exception:
        from pypdf import PdfReader
        reader = PdfReader(str(pdf_path))
        return "\n".join((p.extract_text() or "") for p in reader.pages)

rows = []
for fp in pdfs:
    txt = extract_pdf_text(fp)
    txt = "\n".join(line.strip() for line in txt.splitlines() if line.strip())
    rows.append({"source": "lopd_rgpd", "file": fp.name, "text": txt})

with out_fp.open("w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("WROTE:", out_fp.name, "| DOCS:", len(rows), "| CHARS:", sum(len(r["text"]) for r in rows))

PDFS: ['LOPDGDD_LO_3_2018_BOE.pdf', 'RGPD_AEPD_consolidado_ES.pdf']
WROTE: lopd_rgpd_texts.jsonl | DOCS: 2 | CHARS: 641816


In [26]:
# 11) Sanity check: lopd_rgpd_texts.jsonl (docs + preview)

import json

fp = OUT / "lopd_rgpd_texts.jsonl"
print("EXISTS:", fp.exists(), fp)

# contar docs
n = sum(1 for _ in fp.open("r", encoding="utf-8"))
print("DOCS:", n)

# mostrar 1 doc (primero)
with fp.open("r", encoding="utf-8") as f:
    row = json.loads(f.readline())

print("KEYS:", list(row.keys()))
print("SOURCE:", row.get("source"))
print("FILE:", row.get("file"))
txt = (row.get("text") or "").replace("\n", " ")
print("TEXT_LEN:", len(row.get("text") or ""))
print("PREVIEW:", txt[:500])

EXISTS: True /Users/danielocando/IA/KeepCoding/BOOTCAMP IA/NormaBot TFM BOOTCAMP IA IV/NormaBoot_Data/processed/chunks_legal/lopd_rgpd_texts.jsonl
DOCS: 2
KEYS: ['source', 'file', 'text']
SOURCE: lopd_rgpd
FILE: LOPDGDD_LO_3_2018_BOE.pdf
TEXT_LEN: 253878
PREVIEW: BOLETÍN OFICIAL DEL ESTADO Núm. 294 Jueves 6 de diciembre de 2018 Sec. I.   Pág. 119788 I. DISPOSICIONES GENERALES JEFATURA DEL ESTADO 16673 Ley Orgánica 3/2018, de 5 de diciembre, de Protección de Datos Personales y garantía de los derechos digitales. FELIPE VI REY DE ESPAÑA A todos los que la presente vieren y entendieren. Sabed: Que las Cortes Generales han aprobado y Yo vengo en sancionar la siguiente ley orgánica. ÍNDICE Preámbulo. Título I. Disposiciones generales. Artículo 1. Objeto de la


In [27]:
# 11) Chunking inteligente LOPD/RGPD -> chunks_lopd_rgpd_structured.jsonl

import json, re, hashlib
from pathlib import Path

# --- paths ---
SRC = OUT / "lopd_rgpd_texts.jsonl"
OUT_FP = OUT / "chunks_lopd_rgpd_structured.jsonl"

print("IN EXISTS:", SRC.exists(), SRC.name)

# --- helpers ---
def clean_lines(text: str):
    lines = []
    for ln in text.splitlines():
        ln = ln.replace("\ufeff", "").strip()
        if ln:
            lines.append(ln)
    return lines

# patrones (tolerantes)
RE_TITULO = re.compile(r"^\s*T[ÍI]TULO\s+([IVXLCDM]+)\b\.?\s*(.*)$", re.IGNORECASE)
RE_CAPIT  = re.compile(r"^\s*CAP[ÍI]TULO\s+([IVXLCDM]+)\b\.?\s*(.*)$", re.IGNORECASE)
RE_SECC   = re.compile(r"^\s*SECCI[ÓO]N\s+([IVXLCDM]+)\b\.?\s*(.*)$", re.IGNORECASE)
RE_ART    = re.compile(r"^\s*Art[íi]culo\s+(\d+)\.?\s*(.*)$", re.IGNORECASE)

def chunk_by_structure(text: str):
    """
    Devuelve lista de chunks estructurados:
    cada vez que aparece un Artículo, corta chunk (incluye encabezados previos).
    Si no hay artículos, devuelve 1 chunk grande.
    """
    lines = clean_lines(text)

    cur = {"titulo": None, "capitulo": None, "seccion": None, "articulo": None}
    buf = []
    out = []

    def flush():
        nonlocal buf
        if not buf:
            return
        chunk_text = "\n".join(buf).strip()
        if chunk_text:
            out.append({"meta": cur.copy(), "text": chunk_text})
        buf = []

    seen_article = False

    for ln in lines:
        m = RE_TITULO.match(ln)
        if m:
            cur["titulo"] = (m.group(1) + (" " + m.group(2) if m.group(2) else "")).strip()
            buf.append(ln)
            continue

        m = RE_CAPIT.match(ln)
        if m:
            cur["capitulo"] = (m.group(1) + (" " + m.group(2) if m.group(2) else "")).strip()
            buf.append(ln)
            continue

        m = RE_SECC.match(ln)
        if m:
            cur["seccion"] = (m.group(1) + (" " + m.group(2) if m.group(2) else "")).strip()
            buf.append(ln)
            continue

        m = RE_ART.match(ln)
        if m:
            # si ya veníamos acumulando un artículo anterior, flush para separar
            if seen_article:
                flush()
            seen_article = True
            cur["articulo"] = m.group(1)
            buf.append(ln)
            continue

        buf.append(ln)

    flush()
    if not out:
        out = [{"meta": cur.copy(), "text": "\n".join(lines)}]
    return out

def make_id(payload: dict) -> str:
    s = json.dumps(payload, ensure_ascii=False, sort_keys=True)
    return hashlib.md5(s.encode("utf-8")).hexdigest()

total_docs = 0
total_chunks = 0

with SRC.open("r", encoding="utf-8") as fin, OUT_FP.open("w", encoding="utf-8") as fout:
    for line in fin:
        row = json.loads(line)
        total_docs += 1

        source = row.get("source", "lopd_rgpd")
        file_ = row.get("file")
        text = row.get("text", "")

        chunks = chunk_by_structure(text)

        for i, ch in enumerate(chunks):
            out_row = {
                "id": None,
                "source": source,
                "file": file_,
                "doc_title": file_,  
                "doc_date": None,     
                "titulo": ch["meta"].get("titulo"),
                "capitulo": ch["meta"].get("capitulo"),
                "seccion": ch["meta"].get("seccion"),
                "articulo": ch["meta"].get("articulo"),
                "chunk_index": i,
                "text": ch["text"],
            }
            out_row["id"] = make_id(out_row)

            fout.write(json.dumps(out_row, ensure_ascii=False) + "\n")
            total_chunks += 1

print("OUT WRITTEN:", total_chunks, OUT_FP.name, "| DOCS:", total_docs)

IN EXISTS: True lopd_rgpd_texts.jsonl
OUT WRITTEN: 330 chunks_lopd_rgpd_structured.jsonl | DOCS: 2


In [28]:
# 12) MERGE final: (BOE+EU+AESIA) chunks_final.jsonl  +  (LOPD/RGPD) chunks_lopd_rgpd_structured.jsonl
import json
from pathlib import Path

fp_a = OUT / "chunks_final.jsonl"
fp_b = OUT / "chunks_lopd_rgpd_structured.jsonl"
fp_out = OUT / "chunks_final_all_sources.jsonl"

print("A exists:", fp_a.exists(), fp_a.name)
print("B exists:", fp_b.exists(), fp_b.name)

n_out = 0
with fp_out.open("w", encoding="utf-8") as fout:
    # A
    with fp_a.open("r", encoding="utf-8") as fa:
        for line in fa:
            if line.strip():
                fout.write(line if line.endswith("\n") else line + "\n")
                n_out += 1
    # B
    with fp_b.open("r", encoding="utf-8") as fb:
        for line in fb:
            if line.strip():
                fout.write(line if line.endswith("\n") else line + "\n")
                n_out += 1

print("OUT WRITTEN:", n_out, fp_out.name)

A exists: True chunks_final.jsonl
B exists: True chunks_lopd_rgpd_structured.jsonl
OUT WRITTEN: 515 chunks_final_all_sources.jsonl
